In [13]:
import os
os.getcwd()
import pandas as pd

# Load full corpus
df = pd.read_csv("/Users/divyajyoti/dev/grad/NLP/ANLP-ESG/data/processed/apple_layout_all/corpus_pages_cleaned.csv")

df_amz = df.copy()

In [14]:
import re
import pandas as pd

def clean_page_text(text, patterns):
    """
    Remove all regex patterns from a given text.
    Patterns must be a list of regex strings.
    """
    if pd.isna(text):
        return text
    
    cleaned = text
    for pat in patterns:
        cleaned = re.sub(pat, " ", cleaned, flags=re.IGNORECASE | re.DOTALL)
    # collapse extra whitespace
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned



In [15]:
import re
import pandas as pd

def final_super_clean(text):
    if pd.isna(text):
        return text

    # 1. Remove the classic PDF replacement character
    text = text.replace("�", " ")

    # 2. Remove the explicit weird control chars seen in Apple reports
    text = text.replace("\x08", " ")   # BACKSPACE
    text = text.replace("\x05", " ")   # ENQUIRY

    # 3. Remove invisible Unicode characters (safe)
    text = re.sub(r"[\u200B-\u200D\uFEFF\u00AD]", " ", text)

    # 4. Remove ALL ASCII control characters (0x00–0x1F, 0x7F)
    #    This catches any remaining PDF junk
    text = re.sub(r"[\x00-\x1F\x7F]", " ", text)

    # 5. DO NOT remove non-ASCII (we keep em-dash, accents etc.)

    # 6. Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [16]:
df["page_text_final"] = df["page_text_final"].astype(str).apply(final_super_clean)


In [17]:
import re

patterns = [
    # ===========================================
    # 0. Generic Apple headers / ribbons (multi-year)
    # ===========================================
    r"Our People Environment Introduction Apple’s 20\d{2}",
    r"Appendix\s+Governance\s+Communities\s+Suppliers\s+Customers",
    r"Engagement\s+and\s+Advocacy\s+Introduction\s+Contents",
    r"Engagement\s+and\s+Advocacy\s+Environmental\s+Initiatives",
    r"Environmental\s+Initiatives\s+Introduction\s+Contents\s+Appendix",

    # ===========================================
    # 1. Green Bond / Environmental Progress Report navigation (2023–2025)
    # ===========================================
    r"Appendix\s+Green\s+Bond\s+Impact\s+Report\s+Data\s+\d+\s+2025\s+Environmental\s+Progress\s+Report\s+Environmental\s+Initiatives",
    r"Introduction\s+Contents\s+Appendix\s+Green\s+Bond\s+Impact\s+Report\s+\d+\s+2025\s+Environmental\s+Progress\s+Report\s+Data",
    r"Introduction\s+Contents\s+Appendix\s+Data\s+\d+\s+2025\s+Environmental\s+Progress\s+Report\s+Green\s+Bond\s+Impact\s+Report",
    r"Introduction\s+Contents\s+Green\s+Bond\s+Impact\s+Report\s+Data\s+\d+\s+2025\s+Environmental\s+Progress\s+Report\s+Appendix",

    # Generalized versions (all years)
    r"Appendix\s+Green\s+Bond\s+Impact\s+Report\s+Data\s+\d+\s+20\d{2}\s+Environmental\s+Progress\s+Report(?:\s+Environmental\s+Initiatives)?",
    r"Introduction\s+Contents\s+Green\s+Bond\s+Impact\s+Report\s+\d+\s+20\d{2}\s+Environmental\s+Progress\s+Report",
    r"Introduction\s+Contents\s+Appendix\s+Green\s+Bond\s+Impact\s+Report\s+\d+\s+20\d{2}\s+Environmental\s+Progress\s+Report\s+Data",
    r"Introduction\s+Contents\s+Appendix\s+Data\s+\d+\s+20\d{2}\s+Environmental\s+Progress\s+Report\s+Green\s+Bond\s+Impact\s+Report",

    # ===========================================
    # 2. Year-Specific Navigation Ribbons (2023, 2024)
    # ===========================================
    r"2024\s+Environmental\s+Progress\s+Report\s+\d+\s+Engagement\s+and\s+Advocacy\s+Environmental\s+Initiatives\s+Data\s+Appendix\s+Introduction\s+Contents\s+Introduction",
    r"2023\s+Environmental\s+Progress\s+Report\s+\d+\s+Climate\s+Change\s+Resources\s+Smarter\s+Chemistry\s+Engagement\s+and\s+Advocacy\s+Appendix",

    # ===========================================
    # 3. 2023 EPR Navigation Patterns (with invisible control char)
    # Handles "Report<control> 30", "Report<control> 113", etc.
    # ===========================================
    r"2023\s+Environmental\s+Progress\s+Report\D+\d+.*Climate\s+Change.*Smarter\s+Chemistry.*Engagement\s+and\s+Advocacy.*Appendix.*Introduction",
    r"2023\s+Environmental\s+Progress\s+Report\D+\d+.*Resources.*Smarter\s+Chemistry.*Engagement\s+and\s+Advocacy.*Appendix.*Introduction.*Climate\s+Change",
    r"2023\s+Environmental\s+Progress\s+Report\D+\d+.*Climate\s+Change.*Resources.*Smarter\s+Chemistry.*Engagement\s+and\s+Advocacy.*Appendix.*Introduction",
    r"2023\s+Environmental\s+Progress\s+Report\D+\d+.*Climate\s+Change.*Resources.*Engagement\s+and\s+Advocacy.*Appendix.*Introduction.*Smarter\s+Chemistry",
    r"2023\s+Environmental\s+Progress\s+Report\D+\d+.*Resources.*Smarter\s+Chemistry.*Engagement\s+and\s+Advocacy.*Appendix.*Introduction.*Climate\s+Change",

    # ===========================================
    # 4. ESG Report (All Years)
    # ===========================================
    r"\bESG\s+Report\s+\d+\b",
    r"Apple.?s\s+20\d{2}\s+ESG\s+Report\s+\d+\b",

    # ===========================================
    # 5. Assurance Statements / Third-party verification
    # ===========================================
    r"ASSURANCE\s+STATEMENT\s+Introduction\s+and\s+objectives\s+of",
    r"Apple,\s*Inc\.\s+Bureau\s+Veritas\s+procedures",
    r"of\s+Assurance\s+Client:\s+Apple\s+Inc\.\s+\d+",
    r"Apex\s+Companies,\s+LLC\s+of\s+\d+",
    r"CERT-\d+\s+WM:\s+14001\s+2015\s+INAB\s*\(\d+\)",

    # ===========================================
    # 6. Classic Apple Appendix Headers (2015–2017)
    # ===========================================
    r"Appendix\s+A\s+Apple’s\s+Operations",
    r"Appendix\s+B\s+Data\s+Center\s+Energy",
    r"Appendix\s+B\s+Data\s+Center\s+Energy\s+Supplement",
    r"Appendix\s+C\s+Assurance\s+and\s+Review",
    r"Appendix\s+D\s+Environmental\s+Health\s+and",
    r"Apple,\s*Inc\.\s+Our\s+Conclusion\s+Based",
    r"Safety\s+Policy\s+Statement\s+Mission\s+Statement\s+Apple",
    r"These\s+criteria\s+are\s+also\s+fundamental\s+to",
    r"Report\s+Notes:\s+•\s+This\s+report\s+is",

    # ===========================================
    # 7. Environmental Responsibility (older Apple)
    # ===========================================
    r"Environmental\s+Responsibility\s+at\s+Apple\s+Climate\s+Change",
    r"Environmental\s+Responsibility\s+at\s+Apple\s+To\s+ask",

    # ===========================================
    # 8. Footer / tail-end boilerplate (end of pages)
    # ===========================================
    r"suppliers\s+covering\s+95\s+percent\.$",
    r"environmental\s+requirements\s+Sustainable\s+materials\.$",
    r"Reliability\s+Engineering\s+Reliability\s+Engineering$",
    r"products\s+made\s+with\s+smarter\s+materials\.$",
    r"data\s+centers,\s+and\s+retail\s+stores\.$",
    r"data\s+that\s+are\s+not\s+available\.$",
    r"not\s+add\s+up\s+due\s+to\s+rounding\.$",
    r"energy\s+reliability\s+against\s+local\s+grid\s+outages\.$",
    r"matched\s+with\s+100\s+percent\s+clean\s+energy\.$",
    r"servers\s+with\s+outside\s+air\s+whenever\s+possible\.$",
    r"use\s+to\s+charge\s+their\s+Apple\s+devices\.$",
    r"Further\s+updated\s+information\s+at:.*$",

    # ===========================================
    # 9. Remove ONLY the leading page number (keep the line)
    # ===========================================
    r"^\s*\d+\s+",
]


In [18]:
patterns += [

    # =========================================
    # Apple 2022 – Giant Navigation Ribbon
    # =========================================
    r'^Apple.?s\s+2022\s+Our\s+People\s+Environment\s+Introduction\s+Contents.*?(?=[A-Z])',
    r'^Apple.?s\s+20\d{2}\s+Our\s+People\s+Environment\s+Introduction.*?(?=[A-Z])',

    # =========================================
    # Apple 2021/2022 – Environmental Progress Report Ribbons
    # =========================================
    r'^Environmental\s+Progress\s+Report\s+\d+\s+Introduction\s+Climate\s+Change\s+Resources\s+Smarter\s+Chemistry\s+Appendix\s*',
    r'^Environmental\s+Progress\s+Report\s+\d+\s+(Introduction|Climate\s+Change|Resources|Smarter\s+Chemistry|Appendix)(\s+[A-Za-z]+)*\s*',

    # =========================================
    # Apple 2022 – Massive Table-of-Contents Ribbon
    # Remove until actual section start (Letter from Tim Cook, At Apple, etc.)
    # =========================================
    r'^Apple.?s\s+2022\s+.*?(Letter\s+from\s+Tim\s+Cook|At\s+Apple|Report\s+highlights)',

    # =========================================
    # Leading page numbers (but keep sentence)
    # "23 Introduction Climate Change ..." -> "Introduction Climate Change ..."
    # =========================================
    r'^\s*\d+\s+(?=[A-Z])',

    # =========================================
    # Errant CSV commas at start (,2022,1,)
    # =========================================
    r'^,\s*\d{4},\s*\d+\s*',
]


In [19]:
df_amz["page_text_final_clean"] = df_amz["page_text_final"].apply(
    lambda x: clean_page_text(x, patterns)
)


In [20]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 600)
pd.set_option('display.max_colwidth', None)
df_amz[["page", "page_text_final", "page_text_final_clean"]]

,page,page_text_final,page_text_final_clean
0,0,"Environmental Responsibility Report 2017 Progress Report, Covering Fiscal Year 2016 We are now protecting and creating enough sustainably managed forests in China and the United States to cover all of Apple’s product packaging needs.","Environmental Responsibility Report 2017 Progress Report, Covering Fiscal Year 2016 We are now protecting and creating enough sustainably managed forests in China and the United States to cover all of Apple’s product packaging needs."
1,1,"Environmental Responsibility at Apple Climate Change Resources Safer Materials Better Together Appendix A Apple’s Operations: Environmental Data • Facilities Environmental Key Performance Indicators • Scopes 1 and 2 Building Carbon Emissions • FY2016 Natural Gas and Electricity Use Appendix B Data Center Energy Supplement Appendix C Assurance and Review Statements • Corporate energy, carbon, waste and water data • Product life cycle carbon footprint • Fiber footprint Appendix D Environmental Health and Safety Policy Statement","Resources Safer Materials Better Together : Environmental Data • Facilities Environmental Key Performance Indicators • Scopes 1 and 2 Building Carbon Emissions • FY2016 Natural Gas and Electricity Use Supplement Statements • Corporate energy, carbon, waste and water data • Product life cycle carbon footprint • Fiber footprint Safety Policy Statement"
2,2,"Environmental Responsibility at Apple To ask less of the planet, we’re asking more of ourselves. Can we power a global business with the sun, wind, and water? Can we get 100 percent of our supply chain to move to 100 percent renewable energy? Can we one day stop mining the earth altogether? Can we use only 100 percent recycled and responsibly sourced paper in our packaging? Can we improve on the world’s best materials? That’s Apple: asking bold questions, tackling big problems, and relentlessly innovating to solve them. The approach also lies at the heart of our global environmental strategy—driving us to make not just the best products in the world, but the best products for the world. Our work is led by Lisa Jackson, Apple’s Vice President of Environment, Policy and Social Initiatives, reporting directly to CEO Tim Cook. The Office of Environment, Policy and Social Initiatives works with teams across Apple to set strategy, engage stakeholders, and communicate progress. Our integrated approach means that decisions about Apple values, including environment, are reviewed and supported at the highest levels of the company. Apple’s 2017 Environmental Responsibility Report, covering fiscal year 2016, is our tenth annual update. It highlights the ambitions we have set for ourselves and our progress toward meeting them. We remain focused on three priorities through which we—and our stakeholders—believe Apple can make the biggest difference: • Reduce our impact on climate change by using renewable energy sources and driving energy efficiency in our products and facilities. • Conserve precious resources so we all can thrive. • Pioneer the use of safer materials in our products and processes. We still have a long way to go and a lot to learn. But we’re making tremendous progress. In 2016, 96 percent of the electricity used at our global facilities came from renewable energy, reducing our carbon emissions by nearly 585,000 metric tons. We’re 100 percent renewable in 24 countries—and all of Apple’s data centers.","less of the planet, we’re asking more of ourselves. Can we power a global business with the sun, wind, and water? Can we get 100 percent of our supply chain to move to 100 percent renewable energy? Can we one day stop mining the earth altogether? Can we use only 100 percent recycled and responsibly sourced paper in our packaging? Can we improve on the world’s best materials? That’s Apple: asking bold questions, tackling big problems, and relentlessly innovating to solve them. The approach also lies at the heart of our global environ

In [21]:
df_amz.shape

(949, 5)

In [22]:
df_amz = df_amz.dropna(
    subset=["page_text_final", "page_text_final_clean"]
).copy()
df_amz = df_amz[df_amz["page_text_final_clean"].notna()].copy()


In [23]:
df_amz.shape

(919, 5)

In [24]:
df_amz["page_text_final"] = df_amz["page_text_final_clean"]
df_amz = df_amz.drop(columns=["page_text_final_clean"])
df_amz = df_amz.rename(columns={"page_text_final": "page_text"})
df_amz.to_csv(
    "/Users/divyajyoti/dev/grad/NLP/ANLP-ESG/data/processed/apple_layout_all/corpus_pages_clean_final.csv",
    index=False
)

In [27]:
import pandas as pd
import re

# =========================
# 1. Load the CSV
# =========================
input_path = "/Users/divyajyoti/dev/grad/NLP/ANLP-ESG/data/processed/apple_layout_all/corpus_pages_clean_final.csv"
df = pd.read_csv(input_path)


# =========================
# 2. Helper: remove leading page numbers
#    - handles: "23 Intro...", " 23 Intro...", "  23 Intro..."
#    - preserves normal numbers inside sentences.
# =========================
def remove_leading_page_number(text):
    if pd.isna(text):
        return text

    text = str(text)

    # Normalize weird unicode spaces to regular spaces
    text = re.sub(r"[\u00A0\u2000-\u200B\u202F\u205F\u3000]", " ", text)

    # Remove digits at very start of the text:
    # ^\s*\d+\s+  = any leading spaces + page number + at least one space
    # (?=\S)      = ensure something non-space follows (prevents deleting empty lines)
    text = re.sub(r'^\s*\d+\s+(?=\S)', '', text)

    return text


# =========================
# 3. Apply cleaner
# =========================
df["page_text"] = df["page_text"].astype(str).apply(remove_leading_page_number)


# =========================
# 4. Save output
# =========================
output_path = "corpus_pages_clean_final.csv"
df.to_csv(output_path, index=False)

print("Saved cleaned file to:", output_path)


Saved cleaned file to: corpus_pages_clean_final.csv
